# Qu'est-ce que je montre à la caméra ?

Modèle de tri : il décide si l'objet montré à la caméra est

| classe | signification |
|---|---|
| **euro** | une pièce en euro, n'importe quelle valeur (1 cent à 2 €) |
| **piece** | une pièce, mais pas en euro (autre monnaie, pièce antique) |
| **objet** | pas une pièce (bouchon de bouteille, vis, écrou, bouton, objet quelconque) |

Les pièces **euro** pourront ensuite être envoyées au modèle des pays (`model-train.ipynb`).

Dataset : `python build_euro_dataset.py` crée `dataset_euro/` (voir le script pour les sources).

## 0. Configuration

In [ ]:
from collections import Counter
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

ROOT = Path.cwd()  # le notebook est lancé depuis dataset/
DATA = ROOT / "dataset_euro"
RUNS = ROOT / "runs"

TRAIN = True              # False : charge RUNS/RUN_NAME/weights/best.pt sans réentraîner
RUN_NAME = "objet-cls"

# Réglages CPU (Ryzen 5 PRO 5650U). Tâche plus simple que les pays : 224 px suffisent.
MODEL = "yolo11n-cls.pt"
IMGSZ = 224
EPOCHS = 25
BATCH = 32
WORKERS = 4
SEED = 42

DEVICE = 0 if torch.cuda.is_available() else "cpu"
torch.set_num_threads(6)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

CLASSES = sorted(p.name for p in (DATA / "train").iterdir() if p.is_dir())
val_items = [(p, i) for i, c in enumerate(CLASSES) for p in sorted((DATA / "val" / c).glob("*.jpg"))]
y_true = np.array([y for _, y in val_items])
source = np.array([p.stem.split("_")[0] for p, _ in val_items])  # préfixe du nom = source de l'image
print("classes :", CLASSES)
for split in ("train", "val"):
    print(split, {c: len(list((DATA / split / c).glob("*.jpg"))) for c in CLASSES})

## 1. Exploration

In [ ]:
# Nombre d'images par classe et par source
for split in ("train", "val"):
    counts = Counter((p.parent.name, p.stem.split("_")[0]) for p in (DATA / split).glob("*/*.jpg"))
    print(split, dict(sorted(counts.items())))

fig, axes = plt.subplots(len(CLASSES), 8, figsize=(16, 2.2 * len(CLASSES)))
for row, cls in zip(axes, CLASSES):
    for ax, p in zip(row, random.sample(sorted((DATA / "train" / cls).glob("*.jpg")), 8)):
        ax.imshow(Image.open(p)); ax.set_title(f"{cls}\n{p.stem.split('_')[0]}", fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Entraînement

Augmentations ajoutées par `ObjectTrainer` (Ultralytics ignore `degrees` en classification) :
- **rotation 0-360°** ;
- **masque circulaire** une fois sur deux (`coin_transforms.CircleMask`) : fond blanc hors du cercle,
  comme quand on recadre et masque l'objet détecté par la caméra. Le modèle doit reconnaître les deux cas.

In [ ]:
from torchvision import transforms as T
from ultralytics import YOLO
from ultralytics.models.yolo.classify import ClassificationTrainer

from coin_transforms import CircleMask


class ObjectTrainer(ClassificationTrainer):
    def build_dataset(self, img_path, mode="train", batch=None):
        dataset = super().build_dataset(img_path, mode, batch)
        if mode == "train":
            dataset.torch_transforms.transforms[:0] = [CircleMask(p=0.5), T.RandomRotation(180, fill=(255, 255, 255))]
        return dataset


BEST = RUNS / RUN_NAME / "weights" / "best.pt"
if TRAIN:
    model = YOLO(MODEL)
    model.train(
        trainer=ObjectTrainer,
        data=str(DATA),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        workers=WORKERS,
        cache="ram",
        patience=8,
        device=DEVICE,
        seed=SEED,
        project=str(RUNS),
        name=RUN_NAME,
        exist_ok=True,
        fliplr=0.0,
        hsv_h=0.015,
        hsv_s=0.5,
        hsv_v=0.4,
        erasing=0.2,
    )
    BEST = Path(model.trainer.best)

assert BEST.exists(), f"Poids introuvables : {BEST} (mettre TRAIN = True ou corriger RUN_NAME)"
print("Poids utilisés :", BEST)

## 3. Évaluation

Mesurée sur les images de validation **telles quelles** et **en conditions caméra**
(tournées aléatoirement + masque circulaire).

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

from coin_transforms import CircleMask

best = YOLO(BEST)
assert list(best.names.values()) == CLASSES, "ordre des classes différent"


def predict(images):
    return np.stack([r.probs.data.cpu().numpy()
                     for r in best.predict(images, imgsz=IMGSZ, device=DEVICE, stream=True, verbose=False)])


rng = np.random.default_rng(SEED)
mask = CircleMask(p=1.0)
val_plain = [Image.open(p).convert("RGB") for p, _ in val_items]
val_camera = [mask(im.rotate(a, fillcolor=(255, 255, 255))) for im, a in zip(val_plain, rng.uniform(0, 360, len(val_plain)))]
PROBS = {"telles quelles": predict(val_plain), "conditions caméra": predict(val_camera)}
for name, probs in PROBS.items():
    print(f"accuracy {name:16} : {(probs.argmax(1) == y_true).mean():.3f}")

In [ ]:
probs = PROBS["conditions caméra"]
pred = probs.argmax(1)
fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay.from_predictions(y_true, pred, display_labels=CLASSES, normalize="true",
                                        values_format=".2f", cmap="Blues", colorbar=False, ax=ax)
ax.set_title("Matrice de confusion (conditions caméra)"); plt.tight_layout(); plt.show()
print(classification_report(y_true, pred, target_names=CLASSES, digits=3))

# Précision par source : où le modèle se trompe-t-il ? (ex. bouchons pris pour des euros)
print(f"{'source':12} {'classe':6} {'images':>6} {'accuracy':>8}")
for src, cls in sorted(set(zip(source, y_true))):  # francesco contient des pièces ET des objets
    m = (source == src) & (y_true == cls)
    print(f"{src:12} {CLASSES[cls]:6} {m.sum():>6} {(pred[m] == y_true[m]).mean():>8.3f}")

In [ ]:
# Erreurs (conditions caméra)
errors = np.flatnonzero(pred != y_true)
print(f"{len(errors)} erreurs sur {len(y_true)}")
sel = np.random.default_rng(SEED).choice(errors, min(18, len(errors)), replace=False) if len(errors) else []
if len(sel):
    rows = int(np.ceil(len(sel) / 6))
    fig, axes = plt.subplots(rows, 6, figsize=(15, 2.8 * rows), squeeze=False)
    for ax, i in zip(axes.flat, sel):
        ax.imshow(val_camera[i]); ax.axis("off")
        ax.set_title(f"vrai : {CLASSES[y_true[i]]} ({source[i]})\nprédit : {CLASSES[pred[i]]} {probs[i].max():.0%}",
                     fontsize=8, color="crimson")
    for ax in axes.flat[len(sel):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()

In [ ]:
# Choix du seuil "euro" : on ne considère l'objet comme une pièce en euro
# que si P(euro) >= seuil. Plus le seuil est haut, moins on accepte d'objets par erreur,
# mais plus on rate de vraies pièces en euro.
euro = CLASSES.index("euro")
is_euro = y_true == euro
print(f"{'seuil':>5} {'euros acceptés':>15} {'non-euros acceptés par erreur':>30}")
for t in (0.3, 0.5, 0.7, 0.8, 0.9):
    accepted = probs[:, euro] >= t
    print(f"{t:>5} {accepted[is_euro].mean():>15.1%} {accepted[~is_euro].mean():>30.1%}")

## 4. Nettoyage du dataset

Certaines sources contiennent des erreurs d'étiquette : Francesco/coins-1apki contient par exemple des pièces
**en euro** (surtout espagnoles) rangées comme « piece ». Le modèle entraîné les repère : ce sont les images
d'une autre classe qu'il classe **euro** avec une forte confiance (et inversement).

1. Regarder la grille ci-dessous.
2. Copier dans `exclusions_euro.txt` les noms des images **vraiment** mal étiquetées.
3. Relancer `python build_euro_dataset.py`, puis réentraîner.

In [ ]:
SEUIL = 0.8  # confiance minimale du modèle pour signaler une image
suspects = []
for split in ("train", "val"):
    paths = sorted((DATA / split).glob("*/*.jpg"))
    labels = np.array([CLASSES.index(p.parent.name) for p in paths])
    for start in range(0, len(paths), 256):
        chunk = paths[start:start + 256]
        p = predict([str(x) for x in chunk])
        for path, label, pr in zip(chunk, labels[start:start + 256], p):
            if pr.argmax() != label and pr.max() >= SEUIL and "euro" in (CLASSES[label], CLASSES[pr.argmax()]):
                suspects.append((pr.max(), path, CLASSES[pr.argmax()]))
suspects.sort(reverse=True)
print(f"{len(suspects)} images suspectes (euro <-> autre classe, confiance >= {SEUIL})")

cols = 6
for start in range(0, min(len(suspects), 60), cols * 5):  # par pages de 30
    page = suspects[start:start + cols * 5]
    rows = int(np.ceil(len(page) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(15, 3 * rows), squeeze=False)
    for ax, (conf, path, pred_cls) in zip(axes.flat, page):
        ax.imshow(Image.open(path)); ax.axis("off")
        ax.set_title(f"{path.stem}\nrangée : {path.parent.name} | prédit : {pred_cls} {conf:.0%}", fontsize=7)
    for ax in axes.flat[len(page):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()

# noms à copier dans exclusions_euro.txt après vérification visuelle
print("\n".join(path.stem for _, path, _ in suspects[:60]))

## 5. Export ONNX

Produit `runs/objet-cls/weights/best.onnx` (utilisable sans PyTorch).

In [ ]:
onnx_path = best.export(format="onnx", imgsz=IMGSZ)
print("Exporté :", onnx_path)